# Federated Learning Based CNN for Neonatal Disease Classification (0–17 Days)

In [1]:
!pip install tensorflow pandas numpy scikit-learn matplotlib seaborn

   ---------------------------------------- 0.0/350.9 MB ? eta -:--:--
   ---------------------------------------- 0.2/350.9 MB 4.5 MB/s eta 0:01:19
   ---------------------------------------- 0.4/350.9 MB 4.5 MB/s eta 0:01:19
   ---------------------------------------- 0.7/350.9 MB 5.7 MB/s eta 0:01:02
   ---------------------------------------- 1.1/350.9 MB 6.2 MB/s eta 0:00:57
   ---------------------------------------- 1.4/350.9 MB 6.4 MB/s eta 0:00:55
   ---------------------------------------- 1.8/350.9 MB 6.9 MB/s eta 0:00:51
   ---------------------------------------- 2.2/350.9 MB 6.8 MB/s eta 0:00:52
   ---------------------------------------- 2.3/350.9 MB 6.3 MB/s eta 0:00:56
   ---------------------------------------- 2.7/350.9 MB 6.6 MB/s eta 0:00:54
   ---------------------------------------- 3.1/350.9 MB 6.8 MB/s eta 0:00:52
   ---------------------------------------- 3.5/350.9 MB 7.0 MB/s eta 0:00:50
   ---------------------------------------- 3.9/350.9 MB 7.1 MB/s eta 0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
streamlit 1.32.0 requires protobuf<5,>=3.20, but you have protobuf 7.34.0 which is incompatible.


In [2]:
import pandas as pd
import numpy as np
import tensorflow as tf
from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from tensorflow.keras.utils import to_categorical
import matplotlib.pyplot as plt
import seaborn as sns


[TensorFlow DLL Diagnostic] Analyzing: c:\Users\Asus\anaconda3\Lib\site-packages\tensorflow\python\_pywrap_tensorflow_internal.pyd
[Error] Failed to load _pywrap_tensorflow_common.dll: INITIALIZATION FAILED (0x45A) - The DLL's DllMain returned false.
    Hint: This often happens if your CPU lacks required instructions (like AVX/AVX2)
    or if the Microsoft Visual C++ Redistributable is outdated/missing.


ImportError: Traceback (most recent call last):
  File "c:\Users\Asus\anaconda3\Lib\site-packages\tensorflow\python\pywrap_tensorflow.py", line 74, in <module>
    from tensorflow.python._pywrap_tensorflow_internal import *
ImportError: DLL load failed while importing _pywrap_tensorflow_internal: A dynamic link library (DLL) initialization routine failed.


Failed to load the native TensorFlow runtime.
See https://www.tensorflow.org/install/errors for some common causes and solutions.
If you need help, create an issue at https://github.com/tensorflow/tensorflow/issues and include the entire stack trace above this error message.

## Load Dataset

In [ ]:
df = pd.read_csv('newborn_health_monitoring_with_risk.csv')
df.head()

## Create Multiclass Disease Labels

In [ ]:
def classify_disease(row):
    if row['jaundice_level_mg_dl'] > 12:
        return 'Jaundice'
    elif row['temperature_c'] > 38 and row['heart_rate_bpm'] > 160:
        return 'Sepsis'
    elif row['oxygen_saturation'] < 92:
        return 'Respiratory Distress'
    elif row['temperature_c'] < 36.5:
        return 'Hypothermia'
    elif row['milk_intake_ml'] < 30:
        return 'Hypoglycemia'
    else:
        return 'Healthy'

df['disease'] = df.apply(classify_disease, axis=1)
df['disease'].value_counts()

## Prepare Data

In [ ]:
df = df.drop(['baby_id','name','date','risk_level'], axis=1)
X = df.drop('disease', axis=1)
y = df['disease']

encoder = LabelEncoder()
y_encoded = encoder.fit_transform(y)
y_cat = to_categorical(y_encoded)

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)
X_cnn = np.expand_dims(X_scaled, axis=2)

## Train Test Split

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(X_cnn, y_cat, test_size=0.2, random_state=42)

## CNN Model

In [ ]:
def create_cnn(input_shape, num_classes):
    model = tf.keras.Sequential([
        tf.keras.layers.Conv1D(32,3,activation='relu',input_shape=input_shape),
        tf.keras.layers.Conv1D(64,3,activation='relu'),
        tf.keras.layers.Flatten(),
        tf.keras.layers.Dense(128,activation='relu'),
        tf.keras.layers.Dropout(0.3),
        tf.keras.layers.Dense(num_classes,activation='softmax')
    ])
    model.compile(optimizer='adam', loss='categorical_crossentropy', metrics=['accuracy'])
    return model

## Train Local Model

In [ ]:
input_shape = (X_train.shape[1],1)
num_classes = y_cat.shape[1]

model = create_cnn(input_shape,num_classes)
history = model.fit(X_train, y_train, epochs=10, batch_size=32, validation_data=(X_test,y_test))

## Evaluate Model

In [ ]:
loss, accuracy = model.evaluate(X_test,y_test)
print('Test Accuracy:', accuracy)